In [ ]:
%run ../../langchain_common.py

In [ ]:
from langchain.tools import ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending - echo the body so the model (and final AIMessage) reflects edits
    return f"Email sent with body: {body}"

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def initialize_new_agent(temperature: float | None = None):
    
    class EmailState(AgentState):
        email: str

    pg_checkpointer = create_pg_checkpointer()

    # Optionally override the model temperature. temperature lives on the model,
    # so bind it here rather than passing it to invoke().
    model = llm_noreason if temperature is None else llm_noreason.bind(temperature=temperature)

    agent = create_agent(
        model=model,
        tools=[read_email, send_email],
        state_schema=EmailState,
        checkpointer=pg_checkpointer,
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "read_email": False,
                    "send_email": True,
                },
                description_prefix="Tool execution requires approval",
            ),
        ],
    )
    return agent

agent = initialize_new_agent()

## Invoking the Agent

The cell below kicks off the agent by calling `agent.invoke()` with:

- A **`HumanMessage`** instructing the agent to read and immediately reply to an email
- The **`email`** field populated in the `EmailState` — this is the email content the `read_email` tool will retrieve from state

The agent will:
1. Call `read_email` (no interrupt configured) → retrieves the email body from state
2. Call `send_email` (interrupt configured) → **pauses here** before sending

Because `HumanInTheLoopMiddleware` is set to interrupt on `send_email`, the agent will **not complete** — it returns a `GraphOutput` containing an `__interrupt__` value that holds the proposed email body for human review before it is sent.

In [ ]:
def invoke_agent(agent, config):
    response = agent.invoke(
        {
            "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
            "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John."
        },
        config=config,
    )
    return response

In [ ]:
config = make_thread_config()

agent = initialize_new_agent(temperature=0.0)
response = invoke_agent(agent, config)

for m in response["messages"]:
    m.pretty_print()

In [ ]:
response["__interrupt__"]

In [ ]:
pp(response["__interrupt__"][0].value)

## Approve

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config, # Same thread ID to resume the paused conversation
)

for m in response["messages"]:
    m.pretty_print()

## Reject

In [ ]:
config = make_thread_config()

agent = initialize_new_agent(temperature=0.5)
response = invoke_agent(agent, config)

In [ ]:

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No!! - Request rejected!."
                }
            ],
        }
    ), 
    config=config,
    )   

In [ ]:

for m in response["messages"]:
    m.pretty_print()

## Edit

In [ ]:
config = make_thread_config()

agent = initialize_new_agent(temperature=1.0)
response = invoke_agent(agent, config)

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        "name": "send_email",  # Tool name to call. # Will usually be the same as the original action.
                        
                        # Arguments to pass to the tool. Updated body will generally come from a UI where 
                        # user has edited the email body
                        "args": {"body": "Let me see what we need to do to reschedule the meeting."},
                    }
                }
            ]
        }
    ), 
    config=config, # Same thread ID to resume the paused conversation
    )   

In [ ]:
for m in response["messages"]:
    m.pretty_print()